In [ ]:
%pip -q install qwen-tts fastapi uvicorn soundfile

In [ ]:
import base64, io, os, secrets, threading
import soundfile as sf
import torch
from fastapi import FastAPI, Header, HTTPException
from pydantic import BaseModel, Field
from qwen_tts import Qwen3TTSModel

if not torch.cuda.is_available():
    raise RuntimeError("请在 Kaggle 设置中启用 GPU")
model = Qwen3TTSModel.from_pretrained(
    "Qwen/Qwen3-TTS-12Hz-1.7B-Base", device_map="cuda:0",
    dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
)
from kaggle_secrets import UserSecretsClient
token = UserSecretsClient().get_secret("KAGGLE_TTS_TOKEN")
if not token:
    raise ValueError("请在 Kaggle Secrets 中设置 KAGGLE_TTS_TOKEN")
app = FastAPI(docs_url=None, redoc_url=None)
gpu_lock = threading.Lock()

class Request(BaseModel):
    text: list[str] = Field(min_length=1, max_length=64)
    mode: str
    ref_audio_base64: str = Field(max_length=16_000_000)
    ref_text: str
    language: str = "Chinese"
    temperature: float = Field(default=.7, gt=0, le=2)

@app.post("/tts")
def tts(request: Request, authorization: str = Header(default="")):
    if not secrets.compare_digest(authorization, "Bearer " + token):
        raise HTTPException(401)
    if request.mode != "clone":
        raise HTTPException(400, "需要 clone 模式")
    try:
        sample, rate = sf.read(io.BytesIO(base64.b64decode(request.ref_audio_base64, validate=True)), dtype="float32")
    except (ValueError, RuntimeError):
        raise HTTPException(422, "参考音频无效")
    with gpu_lock, torch.inference_mode():
        prompt = model.create_voice_clone_prompt(ref_audio=(sample, rate), ref_text=request.ref_text)
        waves, rate = model.generate_voice_clone(text=request.text, language=[request.language]*len(request.text),
                                                voice_clone_prompt=prompt, temperature=request.temperature)
    outputs = []
    for waveform in waves:
        buf = io.BytesIO()
        sf.write(buf, waveform, rate, format="WAV", subtype="PCM_16")
        outputs.append({"audio_base64": base64.b64encode(buf.getvalue()).decode("ascii")})
    return {"outputs": outputs}

In [ ]:
import pathlib, platform, re, subprocess, urllib.request
import uvicorn
binary = pathlib.Path("/kaggle/working/cloudflared")
arch = {"x86_64":"amd64", "aarch64":"arm64"}[platform.machine()]
urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-"+arch, binary)
binary.chmod(0o700)
server = uvicorn.Server(uvicorn.Config(app, host="127.0.0.1", port=8000, access_log=False))
threading.Thread(target=server.run, daemon=True).start()
tunnel = subprocess.Popen([str(binary), "tunnel", "--url", "http://127.0.0.1:8000"], stderr=subprocess.PIPE, text=True)
for line in tunnel.stderr:
    match = re.search(r"https://[a-z0-9-]+[.]trycloudflare[.]com", line)
    if match:
        print("KAGGLE_TTS_ENDPOINT_URL=" + match.group(0) + "/tts")
        break
else:
    raise RuntimeError("隧道未建立")